# QSG PMO · Despacho inteligente en Puerto Rico

Este notebook ejecuta el **motor de QSG PMO** sobre un día simulado de cuadrillas HFC y FTTH en Puerto Rico:
el optimizador de rutas contra el motor por reglas, el seguimiento GPS con sus alertas, los materiales del
camión, las zonas y la interpretación de solicitudes de Liberty escritas en texto libre.

**Cómo usarlo:** menú **Entorno de ejecución → Ejecutar todo**. Tarda menos de un minuto y no necesita GPU.

La aplicación completa (tablero de despacho, app de campo, ERPNext) necesita un servidor con Docker, que
Colab no ofrece: se levanta con un clic en **[GitHub Codespaces](https://github.com/codespaces/new?hide_repo_select=true&ref=claude%2Foptimistic-noether-2uct53&repo=1446223164&quickstart=1)**.
Los datos de este notebook son de ejemplo; los algoritmos son los de la app.

In [ ]:
# 1. Descargar QSG PMO (repositorio público) y cargar el motor
import os, subprocess, sys

RAMA = "claude/optimistic-noether-2uct53"  # cambie a "main" cuando el cambio esté aprobado
if not os.path.isdir("PMO"):
    subprocess.run(["git", "clone", "--quiet", "--depth", "1", "--branch", RAMA,
                    "https://github.com/ingcpmo/PMO.git"], check=True)
sys.path.insert(0, os.path.abspath("PMO"))

import qsg_pmo
from qsg_pmo.inteligencia import materiales, solicitudes, territorio, tracking, zonas
from qsg_pmo.inteligencia.optimizacion import Cuadrilla, Opciones, Trabajo, optimizar
from qsg_pmo.assignment.engine import Settings

print(f"QSG PMO {qsg_pmo.__version__} cargado · {len(territorio.PUERTO_RICO)} municipios de Puerto Rico")

## 2. Un día de trabajo simulado

48 órdenes repartidas por la isla (más en el área metro), residenciales y empresariales, FTTH y HFC, con
ventanas AM, PM o todo el día; 10 cuadrillas con base en distintos municipios y habilidades distintas.
Cambie `SEMILLA` para otro día.

In [ ]:
import random
from datetime import datetime

SEMILLA, N_ORDENES, N_CUADRILLAS = 2026, 48, 10
DIA = datetime(2026, 11, 6)

TIPOS = [  # código, habilidades, minutos, segmento, peso
    ("FTTH-INST-RES", {"FTTH-RES"}, 120, "Residencial", 0.27),
    ("FTTH-REP-RES", {"FTTH-RES"}, 90, "Residencial", 0.12),
    ("FTTH-RECON", {"FTTH-RES", "RECONECT"}, 60, "Residencial", 0.12),
    ("HFC-INST-RES", {"HFC-RES"}, 90, "Residencial", 0.14),
    ("HFC-REP-RES", {"HFC-RES"}, 75, "Residencial", 0.10),
    ("HFC-RECON", {"HFC-RES", "RECONECT"}, 45, "Residencial", 0.08),
    ("FTTH-INST-EMP", {"FTTH-EMP"}, 180, "Empresarial", 0.07),
    ("FTTH-REP-EMP", {"FTTH-EMP"}, 120, "Empresarial", 0.05),
    ("HFC-REP-EMP", {"HFC-EMP"}, 120, "Empresarial", 0.05),
]
PESO_ZONA = {"Metro": 0.32, "Este": 0.16, "Norte": 0.14, "Sur": 0.14, "Oeste": 0.13, "Centro": 0.09}
BASES = [
    ("CUA-SJ-01", "San Juan", {"FTTH-RES", "FTTH-EMP", "RECONECT"}),
    ("CUA-CAG-01", "Caguas", {"FTTH-RES", "RECONECT"}),
    ("CUA-PON-01", "Ponce", {"FTTH-RES", "FTTH-EMP", "RECONECT"}),
    ("CUA-MAY-01", "Mayagüez", {"FTTH-RES", "HFC-RES", "RECONECT"}),
    ("CUA-ARE-01", "Arecibo", {"FTTH-RES", "HFC-RES", "RECONECT"}),
    ("CUA-HUM-01", "Humacao", {"HFC-RES", "HFC-EMP", "RECONECT"}),
    ("CUA-BAY-01", "Bayamón", {"HFC-RES", "RECONECT"}),
    ("CUA-CAY-01", "Cayey", {"FTTH-RES", "HFC-RES"}),
    ("CUA-CAR-01", "Carolina", {"FTTH-RES", "HFC-RES", "RECONECT"}),
    ("CUA-AGU-01", "Aguadilla", {"FTTH-RES", "HFC-EMP", "RECONECT"}),
]

def escenario(semilla=SEMILLA, n_ordenes=N_ORDENES, n_cuadrillas=N_CUADRILLAS):
    rnd = random.Random(semilla)
    def elegir(pares):
        x = rnd.random() * sum(p for _, p in pares)
        for valor, peso in pares:
            x -= peso
            if x <= 0:
                return valor
        return pares[-1][0]
    trabajos = []
    for i in range(n_ordenes):
        zona = elegir(list(PESO_ZONA.items()))
        muni = rnd.choice([m for m, v in territorio.PUERTO_RICO.items() if v[3] == zona])
        lat, lng, terreno, _ = territorio.PUERTO_RICO[muni]
        codigo, habilidades, minutos, segmento, _ = elegir([(t, t[4]) for t in TIPOS])
        ws, we = (8, 17) if segmento == "Empresarial" else elegir([((8, 12), .45), ((12, 17), .35), ((8, 17), .2)])
        prioridad = "Alta" if (segmento == "Empresarial" and "REP" in codigo) or rnd.random() < .07 else "Normal"
        trabajos.append(Trabajo(order=f"ORD-{1001 + i}", latitude=lat + (rnd.random() - .5) * .024,
                                longitude=lng + (rnd.random() - .5) * .024, window_start=DIA.replace(hour=ws),
                                window_end=DIA.replace(hour=we), duration_minutes=minutos, skills=set(habilidades),
                                priority=prioridad, zone=zona, segmento=segmento, terreno=terreno))
        trabajos[-1].municipio, trabajos[-1].tipo = muni, codigo
    cuadrillas = []
    for codigo, base, habilidades in BASES[:n_cuadrillas]:
        lat, lng, terreno, zona = territorio.PUERTO_RICO[base]
        cuadrillas.append(Cuadrilla(crew=codigo, latitude=lat, longitude=lng, shift_start=DIA.replace(hour=7),
                                    shift_end=DIA.replace(hour=17), skills=set(habilidades), max_orders=6,
                                    zone=zona, terreno=terreno))
    return trabajos, cuadrillas

trabajos, cuadrillas = escenario()
print(f"{len(trabajos)} órdenes · {sum(t.segmento == 'Empresarial' for t in trabajos)} empresariales · {len(cuadrillas)} cuadrillas")

## 3. Motor por reglas contra optimizado

El motor por reglas es el de la app desde la versión 0.1: asigna una orden a la vez con la misma velocidad en
toda la isla. El optimizador parte de ese plan y lo mejora con el terreno real (la cordillera más lenta, la
lancha a Vieques y Culebra) moviendo, intercambiando e insertando órdenes. Nunca devuelve un plan peor.

In [ ]:
import pandas as pd

r = optimizar(trabajos, cuadrillas, Settings(), opciones=Opciones(tiempo_max_s=5))
ref, opt = r.referencia, r.metricas

tabla = pd.DataFrame({
    "Motor por reglas": [ref.asignadas, ref.sin_asignar, round(ref.km), r.km_por_orden(ref), round(ref.minutos_viaje / 60, 1)],
    "Optimizado": [opt.asignadas, opt.sin_asignar, round(opt.km), r.km_por_orden(opt), round(opt.minutos_viaje / 60, 1)],
}, index=["Órdenes asignadas", "Sin asignar", "Km de manejo", "Km por orden", "Horas de manejo"])
display(tabla)
print(r.resumen())

In [ ]:
# Mapa de las rutas: izquierda el motor por reglas, derecha el optimizado
import matplotlib.pyplot as plt

COSTA = [(-67.16,18.51),(-66.95,18.49),(-66.70,18.48),(-66.45,18.48),(-66.20,18.47),(-66.05,18.47),(-65.90,18.45),
         (-65.70,18.38),(-65.62,18.33),(-65.62,18.22),(-65.73,18.12),(-65.80,18.05),(-65.88,17.98),(-66.02,17.97),
         (-66.15,17.94),(-66.35,17.95),(-66.50,17.97),(-66.62,17.98),(-66.80,17.98),(-66.92,17.94),(-67.10,17.95),
         (-67.20,17.96),(-67.19,18.07),(-67.20,18.20),(-67.26,18.36),(-67.16,18.51)]
por_orden = {t.order: t for t in trabajos}
base = {c.crew: c for c in cuadrillas}

def rutas_de(asignaciones):
    rutas = {}
    for a in sorted(asignaciones, key=lambda a: (a.crew, a.sequence)):
        rutas.setdefault(a.crew, []).append(por_orden[a.order])
    return rutas

fig, ejes = plt.subplots(1, 2, figsize=(16, 4.3))
# El plan por reglas tal como queda con el terreno real (las paradas que no llegan a tiempo se caen)
for eje, titulo, asignaciones in [(ejes[0], "Motor por reglas", r.asignaciones_referencia), (ejes[1], "Optimizado", r.asignaciones)]:
    eje.fill(*zip(*COSTA), color="#eef3f2", edgecolor="#9fb3b0")
    rutas = rutas_de(asignaciones)
    for i, (crew, paradas) in enumerate(sorted(rutas.items())):
        xs = [base[crew].longitude] + [p.longitude for p in paradas]
        ys = [base[crew].latitude] + [p.latitude for p in paradas]
        eje.plot(xs, ys, "-o", ms=3, lw=1.4, color=plt.cm.tab10(i % 10), label=crew)
    asignadas = {a.order for a in asignaciones}
    fuera = [t for t in trabajos if t.order not in asignadas]
    eje.scatter([t.longitude for t in fuera], [t.latitude for t in fuera], marker="x", color="#c0392b", label="Sin asignar")
    eje.set_title(f"{titulo}: {len(asignadas)} asignadas")
    eje.set_aspect(1 / 0.95); eje.axis("off")
fig.legend(*ejes[1].get_legend_handles_labels(), loc="lower center", ncol=6, fontsize=8, frameon=False)
plt.tight_layout(rect=(0, 0.12, 1, 1)); plt.show()

## 4. Seguimiento GPS: alertas para el despachador

Se simula una cuadrilla que se detiene 40 minutos lejos de sus órdenes y deja de enviar señal; el motor de
seguimiento compara su posición con la agenda y genera las alertas que ve el tablero.

In [ ]:
from datetime import timedelta

ahora = DIA.replace(hour=11)
quieta = [tracking.Ping("CUA-CAY-01", DIA.replace(hour=9, minute=50) + timedelta(minutes=5 * i), 18.20, -66.60) for i in range(9)]
agenda = [tracking.Visita("ORD-2001", 18.45, -66.05, "Programada", ventana_fin=DIA.replace(hour=11, minute=10))]
estado = tracking.estado("CUA-CAY-01", quieta, agenda, ahora)
display(pd.DataFrame([{"Alerta": a.tipo, "Severidad": a.severidad, "Mensaje": a.mensaje} for a in estado.alertas]))

## 5. Materiales: recoger neto del camión

Recoger = lo que piden las órdenes del día + 10 % de margen − lo que ya trae el camión. ONT y módems: lo del
día más un repuesto; el resto se devuelve a bodega.

In [ ]:
KITS = {
    "FTTH-INST-RES": [("DROP-FO-50M", 1), ("ONT", 1), ("CONECTOR-SC", 2), ("ROSETA", 1)],
    "FTTH-INST-EMP": [("DROP-FO-50M", 1), ("ONT", 1), ("CONECTOR-SC", 4), ("ROSETA", 1)],
    "FTTH-REP-RES": [("CONECTOR-SC", 2)], "FTTH-REP-EMP": [("CONECTOR-SC", 2), ("ONT", 1)],
    "FTTH-RECON": [("CONECTOR-SC", 2)], "HFC-INST-RES": [("COAX-RG6-30M", 1), ("MODEM", 1), ("CONECTOR-F", 4)],
    "HFC-REP-RES": [("CONECTOR-F", 2)], "HFC-REP-EMP": [("CONECTOR-F", 2), ("MODEM", 1)], "HFC-RECON": [("CONECTOR-F", 4)],
}
necesidad = materiales.necesidades([(a.crew, por_orden[a.order].tipo) for a in r.asignaciones], KITS)
rnd = random.Random(7)
camion = {c: {i: rnd.randint(0, 6) for i in ["DROP-FO-50M", "ONT", "CONECTOR-SC", "ROSETA", "COAX-RG6-30M", "MODEM", "CONECTOR-F"]} for c in necesidad}
plan = materiales.plan_bodega(necesidad, camion, serializados={"ONT", "MODEM"})
fmt = lambda d: ", ".join(f"{q:g} {i}" for i, q in d.items()) or "—"
display(pd.DataFrame([{"Cuadrilla": c, "Recoger en bodega": fmt(p["recoger"]), "Devolver": fmt(p["devolver"])} for c, p in sorted(plan.items())]))

## 6. Zonas: demanda contra capacidad

Minutos de trabajo pedidos por zona contra los turnos de las cuadrillas con base en ella, y qué cuadrilla
conviene mover desde una zona vecina holgada.

In [ ]:
demanda, capacidad, por_zona = {}, {}, {}
for t in trabajos:
    demanda[t.zone] = demanda.get(t.zone, 0) + t.duration_minutes + 25
for c in cuadrillas:
    minutos = (c.shift_end - c.shift_start).total_seconds() / 60
    capacidad[c.zone] = capacidad.get(c.zone, 0) + minutos
    por_zona.setdefault(c.zone, []).append((c.crew, minutos))
balance = zonas.demanda_capacidad(demanda, capacidad, por_zona)
display(pd.DataFrame(balance["zonas"]).T[["demanda_min", "capacidad_min", "ocupacion_pct", "estado", "estado_con_movimientos"]])
for m in balance["movimientos"]:
    print(f"Mover {m['cuadrilla']} de {m['de']} a {m['a']}: {m['motivo']}")

## 7. Solicitudes de Liberty en texto libre

El correo o mensaje se convierte en una orden con tipo de trabajo, prioridad, municipio, zona, fecha y
ventana. Sin modelo de lenguaje: reglas. En la app, Claude u Ollama pueden completar lo que las reglas no
entienden, siempre validado contra el catálogo.

In [ ]:
from datetime import date

catalogo = solicitudes.Catalogo(tipos=[solicitudes.TipoTrabajo(c, c, c.split("-")[0], "Empresarial" if c.endswith("EMP") else "Residencial",
    {"INST": "Instalación", "REP": "Reparación", "RECON": "Reconectorización"}[c.split("-")[1]]) for c, *_ in TIPOS])
mensajes = [
    "Ticket 55120. Cliente: Ferretería Boricua. Dirección: Carr. 2 km 5, Mayagüez. Tel 787-555-2211. El negocio está sin servicio de fibra, mañana de 8 a 12.",
    "Instalación de internet por cable para casa en Humacao, el lunes por la tarde. Tel (939) 555-0199, cuenta 80012345.",
    "Cambio de conector F en el tap, casa en Vieques, urgente.",
]
filas = []
for m in mensajes:
    d = solicitudes.extraer(m, catalogo, date(2026, 11, 5))
    filas.append({k: d.get(k) for k in ["work_type", "priority", "municipality", "zone", "appointment_date", "window_start", "window_end", "phone"]} | {"faltan": ", ".join(d["faltantes"])})
display(pd.DataFrame(filas))

## 8. Simulador interactivo

El mismo motor en JavaScript: cambie el número de órdenes y cuadrillas, active un apagón de LUMA en una zona y
reproduzca el día para ver las alertas y la mejor cuadrilla.

In [ ]:
from IPython.display import HTML
HTML(open("PMO/docs/simulaciones/simulador-despacho.html", encoding="utf-8").read())

## Siguiente paso: la aplicación completa

Para ver el tablero de despacho, la app del técnico y la jornada simulada en vivo, abra
**[QSG PMO en GitHub Codespaces](https://github.com/codespaces/new?hide_repo_select=true&ref=claude%2Foptimistic-noether-2uct53&repo=1446223164&quickstart=1)** (con su cuenta de GitHub; el primer arranque tarda 10 a 15
minutos). Las instrucciones y usuarios de la demostración están en `.devcontainer/LEAME.md`.